# This notebook starts from scratch, performs the following steps:
- Step 1: Loads the base roadway and transit networks, attributes roadway network, adds rail links and nodes to base roadway
- Step 2: Creates v00 scenario with only base roadway, to apply base roadway correction project cards in four folders
- Step 3: Creates v01 scenario with both roadway from step 2 and base transit, to apply base transit correction project cards
- Step 4: Write out v01 scenario (aka the "corrected" base roadway and transit) network 

You would only need to rerun this notebook if you have:
- Rebuilt base roadway or transit network
- Changed the base correction project cards

The project cards applied in this notebok are:
- `BaseCorrections`
- `BaseCorrections2`
- `BaseMissingRoads`
- `BaseAttribute`
- `BaseCorrection_transit`

In [2]:
import os
import sys
import numpy as np

from pyproj import CRS
from pathlib import Path

from projectcard import read_cards

import network_wrangler
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

# from cube_wrangler import Parameters
# from cube_wrangler import util
# from cube_wrangler import roadway
# from cube_wrangler import StandardTransit

c:\Users\USLP095001\code\met_council\met_council_wrangler\.venv\lib\site-packages\network_wrangler\models\roadway\tables.py:41: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  class RoadLinksTable(DataFrameModel):


In [3]:
network_wrangler.setup_logging()

In [4]:
%reload_ext autoreload
%autoreload 2

# remote i/o

In [5]:
# v00 inputs are saved on SharePoint
# https://wsponlinenam.sharepoint.com/:f:/r/Sites/US-MetCNetworkWrangler/Shared%20Documents/Final%20Deliverables/Data/standard_networks/v00?csf=1&web=1&e=ppEBxn
input_dir = os.path.join(r"C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors")
cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
rail_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")
tran_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_network = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")

In [ ]:
# project cards are available in Github Repo `project_card_registry`, branch `Test1_converted_project_cards`
# https://github.com/Metropolitan-Council/project_card_registry/tree/Test1_converted_project_cards/projects
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections")
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads")
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute")
project_card_dir5 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrection_transit")

In [7]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

2026-10-30 13:00:28, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2026-10-30 13:00:28, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


# Step 1 - Load Version00 standard networks

In [12]:
link_file = os.path.join(input_dir, 'link.json')
node_file = os.path.join(input_dir, 'node.geojson')
shape_file = os.path.join(input_dir, 'shape.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

transit_net = load_transit(os.path.join(tran_dir,"standard_transit_network"))

2026-10-30 13:03:41, DEBUG: Reading nodes from C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\node.geojson.
2026-10-30 13:03:41, DEBUG: Estimated read time: 4 seconds.
2026-10-30 13:03:49, DEBUG: Read 417161 nodes from file in 7.36.
2026-10-30 13:03:49, DEBUG: Turning node data into official nodes_df
2026-10-30 13:03:51, INFO: Read 417161 nodes from C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\node.geojson in 9.98.
2026-10-30 13:03:52, INFO: Reading links from C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\link.json.
2026-10-30 13:03:52, DEBUG: Estimated read time: 3 minutes.
2026-10-30 13:04:26, DEBUG: Read 1103378 links in 33.87.
2026-10-30 13:04:26, DEBUG: Creating 110

c:\Users\USLP095001\code\met_council\met_council_wrangler\.venv\lib\site-packages\network_wrangler\transit\io.py:79: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


2026-10-30 13:05:32, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\trips.txt.
2026-10-30 13:05:34, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\stop_times.txt.
2026-10-30 13:05:36, INFO: Initializing frequencies
2026-10-30 13:05:36, DEBUG: Validating + coercing value to frequencies
2026-10-30 13:05:36, DEBUG: PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
2026-10-30 13:05:36, INFO: Initializing routes
2026-10-30 13:05:36, DEBUG: Validating + coercing value to routes
2026-10-30 13:05:36, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2026-10-30 13:05:36, DEBUG: Referencing table 

In [13]:
roadway_net.links_df.shape

(1103378, 55)

In [14]:
roadway_net.nodes_df.shape

(417161, 12)

In [15]:
# Drop loop links
roadway_net.links_df = roadway_net.links_df[roadway_net.links_df.A != roadway_net.links_df.B]

### Attribute the Roadway Network

In [16]:
roadway_net.links_df = roadway_net.links_df.drop('lanes', axis = 1)

In [17]:
# make sure the data types of the boolean columns are correct
roadway_net.links_df['bus_only'] = False

for c in list(set(roadway_net.links_df.columns) & set(metcouncil_parameters.bool_col)):
    roadway_net.links_df[c] = roadway_net.links_df[c].replace(
        {
            np.nan: False,
            "": False,
            "0": False,
            "1": True
        }
    )

In [18]:
r_net = metcouncil_roadway.calculate_number_of_lanes_from_reviewed_network(
    roadway_net=roadway_net,
    parameters=metcouncil_parameters,
)
r_net.links_df.lanes.value_counts()

2026-10-30 13:06:22, INFO: Calculating Number of Lanes as network variable: 'lanes'
2026-10-30 13:06:23, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2026-10-30 13:06:23, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2026-10-30 13:06:23, DEBUG: Calculating Centroid Connectors
2026-10-30 13:06:23, INFO: Calculating Centroid Connector and adding as roadway network variable: centroidconnect
2026-10-30 13:06:23, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2026-10-30 13:06:23, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2026-10-30 13:06:23, DEBUG: Calculating Centroid Connectors using highest TAZ number: 3100
2026-10-30 13:06:23, INFO: Finished calculating c

MemoryError: Unable to allocate 13.0 GiB for an array with shape (1749304901, 1) and data type object

In [ ]:
r_net.links_df.roadway.value_counts()

In [ ]:
r_net = metcouncil_roadway.calculate_assign_group_and_roadway_class_from_reviewed_network(
        roadway_net=r_net,
        parameters=metcouncil_parameters,
)
r_net.links_df.assign_group.value_counts(dropna=False)

In [ ]:
r_net.links_df.roadway_class.value_counts()

### Add Rail links and nodes to the roadway network

In [ ]:
# rail links and nodes need to be added right after attributing the network and before applying any project cards
# this step can't be moved
r_net = metcouncil_roadway.add_rail_links_and_nodes(
    roadway_network = r_net,
    parameters = metcouncil_parameters,
    rail_links_file = os.path.join(rail_dir, 'rail_links.geojson'),
    rail_nodes_file = os.path.join(rail_dir, 'rail_nodes.geojson'),
)

In [ ]:
# check if missing IDs
# rail nodes does not have osm IDs, they have shst IDs
# rail links does not have osm IDs, they have shst IDs

# if node missing shst id
print(r_net.nodes_df.shst_node_id.isnull().sum())
print(r_net.nodes_df.shst_node_id.nunique())
# if node missing osm id
print(r_net.nodes_df.osm_node_id.isnull().sum() + len(r_net.nodes_df[r_net.nodes_df.osm_node_id==""]))
# if node missing osm id
print(r_net.nodes_df.osm_node_id.dtype)
print(r_net.nodes_df[r_net.nodes_df.osm_node_id == 0])
print(r_net.nodes_df.osm_node_id.nunique())
# if node missing model node id
print(r_net.nodes_df.model_node_id.nunique())

# if link missing 
print(r_net.links_df.shstReferenceId.isnull().sum())
print(r_net.links_df.shstReferenceId.nunique())
print(r_net.links_df.model_link_id.nunique())

# if link missing node id
print(r_net.links_df.fromIntersectionId.isnull().sum())
print(r_net.links_df.toIntersectionId.isnull().sum())
print(r_net.links_df.u.isnull().sum())
print(r_net.links_df[r_net.links_df.u == 0])
print(r_net.links_df.v.isnull().sum())
print(r_net.links_df[r_net.links_df.v == 0])

# Step 2 - Create Scenario 00 (with Roadway only, no Transit)
A starting point for applying base roadway correction project cards. This section applied project cards in these folders:
- `BaseAttribute`: applied in scenario 00B
- `BaseCorrections`: applied in scenario 00C
- `BaseMissingRoads`: applied in scenario 00C
- `BaseCorrections2`: applied in scenario 00D

In [ ]:
# We are not adding the base transit to this sceanrio because
# the base transit is built on top of the "corrected" base roadway.
# We will add the base transit after applying the base roadway correction project cards.
base_scenario = {"road_net": r_net}
version_00_scenario = create_scenario(base_scenario = base_scenario)

### Create Scenario 00B
This adds the project cards in the `BaseAttribute` folder, that includes:
walk and bike variables to network, manually calculates external connectors, transit priority, and some roadclass values

In [ ]:
version_00b_scenario = create_scenario(
    base_scenario = version_00_scenario,
    project_card_filepath = project_card_dir4
)

In [ ]:
version_00b_scenario.apply_all_projects()

### Create Scenario 00C
This adds the project cards in the `BaseCorrections` folder and the `BaseMissingRoads` folder 

In [ ]:
version_00c_scenario = create_scenario(
    base_scenario=version_00b_scenario,
    project_card_filepath = project_card_dir
)

In [ ]:
version_00c_scenario.add_project_cards(
     list(
            read_cards(project_card_dir3).values()
        )
)

In [ ]:
version_00c_scenario.apply_all_projects()

### Create Scenario 00D
This adds project cards in the `BaseCorrections2` folder, which are newer attribute clean up cards 

In [ ]:
version_00d_scenario = create_scenario(
    base_scenario=version_00c_scenario,
    project_card_filepath = project_card_dir2
)

In [ ]:
version_00d_scenario.apply_all_projects()

In [ ]:
version_00d_scenario.applied_projects

In [ ]:
version_00d_scenario.road_net.links_df['access'] = version_00d_scenario.road_net.links_df['access'].fillna('')
version_00d_scenario.road_net.links_df['access'] = version_00d_scenario.road_net.links_df['access'].apply(
    lambda x: util.shorten_name(x)
)

In [ ]:
version_00d_scenario.road_net.links_df['name'] = version_00d_scenario.road_net.links_df['name'].fillna('')
version_00d_scenario.road_net.links_df['name'] = version_00d_scenario.road_net.links_df['name'].apply(
    lambda x: util.shorten_name(x)
)

# Step 3 - Create Scenario 01 (include both Roadway and Transit)
This adds transit network to the scenario, and applies transit corrections in `BaseCorrection_transit`

In [ ]:
# add transit network to version 01 scenario
version_01_scenario = create_scenario(
    base_scenario = version_00d_scenario
)
version_01_scenario.transit_net = transit_net

In [ ]:
version_01_scenario = create_scenario(
    base_scenario = version_01_scenario,
    project_card_filepath = project_card_dir5, # transit correction
)
version_01_scenario.transit_net.road_net = version_01_scenario.road_net

In [ ]:
version_01_scenario.apply_all_projects()

In [ ]:
version_01_scenario.applied_projects

# Step 4 - Save out version 01 standard networks - aka the "Corrected Base" Standard Networks.

In [ ]:
version_01_scenario.write(
    os.path.join(net_dir, 'v01', 'standard_networks'),
    name = 'v01',
    roadway_file_format = "geojson",
    transit_file_format = "txt",
    roadway_write = True,
    transit_write = True,
    projects_write = True,
    overwrite = True,
    roadway_convert_complex_link_properties_to_single_field=True
)

# The sections below are Optional

# Optional - Make Travel Model Network

### Add centroid and centroid connectors

In [ ]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_01_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(input_dir, 'standard_networks', 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(input_dir, 'standard_networks', 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(input_dir, 'standard_networks', 'cc_shape.pickle'),
)

### Add Rail access and egress links

In [ ]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters,
    exclude_rail_node_id = [417275, 417255]
)

In [ ]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

In [ ]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum() + len(m_net.links_df[m_net.links_df.shstReferenceId==""]))
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

In [ ]:
m_net.nodes_df.shape

In [ ]:
m_net.nodes_df.columns

In [ ]:
m_net.links_df.columns

# Optional - Write model network as shapefile

In [ ]:
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 'bike', 'walk','roadway',
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'links_v01.shp'),
    output_node_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'nodes_v01.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

# Optional - Write model network for Cube

In [ ]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_network, 'fullnet_v01', 'links.txt'),
    output_node_txt = os.path.join(output_network,  'fullnet_v01','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_network,  'fullnet_v01', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_network,  'fullnet_v01','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_network,  'fullnet_v01',  'make_complete_network_from_fixed_width_file.s'),
)

In [ ]:
version_01_scenario.transit_net.road_net = version_01_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

In [ ]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_network, 'fullnet_v01', "line_name_xwalk.csv")
)    

In [ ]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_network, 'fullnet_v01', "transit.lin"))

In [ ]:
version_01_scenario.road_net.nodes_df.model_node_id.max()

In [ ]:
version_01_scenario.road_net.nodes_df.model_node_id.shape

In [ ]:
m_net.nodes_df.model_node_id.max()

In [ ]:
m_net.nodes_df.model_node_id.shape